In [14]:
# 데이터 생성 : csv파일을 불러옴
import pandas as pd

df = pd.read_csv('../data/csv/sentiment_data.csv')

In [15]:
# 독립, 종속변수 분리
X = df['sentence']
y = df['label']

In [16]:
# 훈련, 테스트 분리
DATA_SIZE = 1000
TRAIN_SIZE = int(DATA_SIZE * 0.8)

X_train, X_test = X[:TRAIN_SIZE], X[TRAIN_SIZE: ]
y_train, y_test = y[:TRAIN_SIZE], y[TRAIN_SIZE: ]

In [17]:
# 형태소 분리 시 모든 형태소를 포함
# 분리한 단어들을 합침
from konlpy.tag import Okt

def get_preprocessing(sentence):
    okt = Okt()
    result = okt.pos(sentence, stem=True) # 문장을 형태소별로 나눔. 단, 원형으로
    words = [word for word, pos in result]
    return " ".join(words)

# X_train과 X_test의 있는 문장들을 get_preprocessing에 적용
X_train = X_train.apply(get_preprocessing)
X_test = X_test.apply(get_preprocessing)

In [18]:
# 벡터화
from tensorflow.keras import layers, models
vectorize_layer = layers.TextVectorization(
	max_tokens = 1000,
	output_mode = "multi_hot"
)
vectorize_layer.adapt(X_train.tolist())

In [19]:
import tensorflow as tf

# 모델 설계
model = models.Sequential([
	layers.Input((1, ), dtype=tf.string),
	vectorize_layer,
	layers.Dense(64, activation='relu'),
	layers.Dense(32, activation='relu'),
	layers.Dense(1, activation='sigmoid') # 출력층
])

In [20]:
# 모델 설정
model.compile(
	optimizer='adam', 
	loss='binary_crossentropy',
	metrics=['accuracy']
)

In [22]:
# 학습
history = model.fit(
	X_train.values, y_train, epochs=100, verbose=1, validation_split=0.2,
	batch_size=32
)

Epoch 1/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - accuracy: 1.0000 - loss: 2.4190e-05 - val_accuracy: 1.0000 - val_loss: 2.0114e-05
Epoch 2/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 1.0000 - loss: 2.3687e-05 - val_accuracy: 1.0000 - val_loss: 1.9740e-05
Epoch 3/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 1.0000 - loss: 2.3204e-05 - val_accuracy: 1.0000 - val_loss: 1.9379e-05
Epoch 4/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 1.0000 - loss: 2.2730e-05 - val_accuracy: 1.0000 - val_loss: 1.9031e-05
Epoch 5/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 1.0000 - loss: 2.2268e-05 - val_accuracy: 1.0000 - val_loss: 1.8671e-05
Epoch 6/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 1.0000 - loss: 2.1823e-05 - val_accuracy: 1.0000 - val_loss: 1.8338e-05
Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 1.0000 - loss: 2.1384e-05 - val_accuracy: 1.0000 - val_loss: 1.8006e-05
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - a

In [34]:
# 평가
_, acc = model.score(X_test, y_test)
print(f'정확도 : {acc}')

AttributeError: 'Sequential' object has no attribute 'score'

In [26]:
# 예측
# "너무 피곤하고 듣고 있는 음악은 별로지만 기분이 좋아요"
# 오늘 날이 좋고 기분이 좋아요
# 듣고 있는 음악이 슬퍼서 우울해요
